Name: Rucha Vikrant Wadhavankar

SRN: PES2UG23CS496

# Unit 3 Assignment: Building a Production Advanced RAG System

In [7]:
%pip install python-dotenv rank-bm25 sentence-transformers langchain langchain-community langchain-google-genai numpy --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [4]:
from dotenv import load_dotenv
import os
import getpass

load_dotenv()

if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API Key: ")

print("Setup complete.")

Enter your Google API Key: ··········
Setup complete.


## Part 1: Document Corpus Setup

Create a corpus of at least 10 documents on AI/ML topics with:
- At least 3 documents on related but distinct sub-topics
- At least 1 document with technical jargon that BM25 would excel at

In [5]:
corpus = [
    "Transformers use self-attention mechanisms to process sequences in parallel, enabling efficient training on long documents.",
    "The multi-head attention mechanism in transformers allows the model to attend to different parts of the input simultaneously.",
    "Positional encoding in transformer architectures encodes the order of tokens since self-attention is inherently position-agnostic.",
    "Gradient descent is an optimization algorithm that iteratively updates weights to minimize the loss function in neural networks.",
    "Stochastic Gradient Descent (SGD) uses randomly selected mini-batches to compute gradients, providing a trade-off between accuracy and speed.",
    "Adaptive learning rate methods like Adam adjust the learning rate for each parameter during training to accelerate convergence.",
    "BERT is a bidirectional transformer encoder trained using masked language modeling where 15% of tokens are randomly masked.",
    "The BM25 algorithm (Best Match 25) ranks documents using TF-IDF with term frequency saturation and document length normalization via IDF calculation.",
    "Neural networks learn by adjusting weights through backpropagation, which computes gradients of the loss with respect to each parameter.",
    "Retrieval Augmented Generation combines a dense retriever with a language model to answer questions using grounded context from a knowledge base.",
]

print(f"Corpus loaded: {len(corpus)} documents")
for i, doc in enumerate(corpus):
    print(f"  [{i}] {doc[:80]}...")

Corpus loaded: 10 documents
  [0] Transformers use self-attention mechanisms to process sequences in parallel, ena...
  [1] The multi-head attention mechanism in transformers allows the model to attend to...
  [2] Positional encoding in transformer architectures encodes the order of tokens sin...
  [3] Gradient descent is an optimization algorithm that iteratively updates weights t...
  [4] Stochastic Gradient Descent (SGD) uses randomly selected mini-batches to compute...
  [5] Adaptive learning rate methods like Adam adjust the learning rate for each param...
  [6] BERT is a bidirectional transformer encoder trained using masked language modeli...
  [7] The BM25 algorithm (Best Match 25) ranks documents using TF-IDF with term freque...
  [8] Neural networks learn by adjusting weights through backpropagation, which comput...
  [9] Retrieval Augmented Generation combines a dense retriever with a language model ...


## Part 2: Hybrid Retriever Implementation

Implement the HybridRetriever class combining BM25 (sparse) + SBERT (dense) with Reciprocal Rank Fusion (RRF).

**Returns:** List of dicts with {doc_id, rrf_score, bm25_rank, sbert_rank, text}

In [8]:
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

class HybridRetriever:

    def __init__(self, corpus: list[str], k: int = 60, sbert_model: str = "sentence-transformers/all-MiniLM-L6-v2"):
        self.corpus = corpus
        self.k = k

        tokenized_corpus = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(tokenized_corpus)

        self.sbert = SentenceTransformer(sbert_model)
        doc_vecs = self.sbert.encode(corpus, convert_to_numpy=True)
        self.doc_vecs = doc_vecs / np.linalg.norm(doc_vecs, axis=1, keepdims=True)

    def retrieve(self, query: str, top_k: int = 5) -> list[dict]:
        bm25_scores = self.bm25.get_scores(query.lower().split())
        bm25_ranked = np.argsort(bm25_scores)[::-1]
        bm25_ranks = {int(doc_id): rank + 1 for rank, doc_id in enumerate(bm25_ranked)}

        query_vec = self.sbert.encode([query], convert_to_numpy=True)[0]
        query_vec = query_vec / np.linalg.norm(query_vec)
        sbert_scores = self.doc_vecs @ query_vec
        sbert_ranked = np.argsort(sbert_scores)[::-1]
        sbert_ranks = {int(doc_id): rank + 1 for rank, doc_id in enumerate(sbert_ranked)}

        rrf_scores = {}
        for doc_id in range(len(self.corpus)):
            rrf_bm25 = 1.0 / (self.k + bm25_ranks[doc_id])
            rrf_sbert = 1.0 / (self.k + sbert_ranks[doc_id])
            rrf_scores[doc_id] = rrf_bm25 + rrf_sbert

        final_ranked = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

        results = [
            {
                "doc_id": doc_id,
                "rrf_score": rrf_scores[doc_id],
                "bm25_rank": bm25_ranks[doc_id],
                "sbert_rank": sbert_ranks[doc_id],
                "text": self.corpus[doc_id]
            }
            for doc_id, _ in final_ranked
        ]

        return results


hybrid_retriever = HybridRetriever(corpus)
print("HybridRetriever initialized with corpus")

test_query = "How does attention work in neural networks?"
results = hybrid_retriever.retrieve(test_query, top_k=3)

print(f"\nTest Query: '{test_query}'")
print("-" * 100)
for r in results:
    print(f"  [doc_{r['doc_id']}] RRF={r['rrf_score']:.6f}, BM25_rank={r['bm25_rank']}, SBERT_rank={r['sbert_rank']}")
    print(f"    → {r['text'][:90]}...")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

HybridRetriever initialized with corpus

Test Query: 'How does attention work in neural networks?'
----------------------------------------------------------------------------------------------------
  [doc_1] RRF=0.032522, BM25_rank=1, SBERT_rank=2
    → The multi-head attention mechanism in transformers allows the model to attend to different...
  [doc_8] RRF=0.032266, BM25_rank=3, SBERT_rank=1
    → Neural networks learn by adjusting weights through backpropagation, which computes gradien...
  [doc_3] RRF=0.031514, BM25_rank=2, SBERT_rank=5
    → Gradient descent is an optimization algorithm that iteratively updates weights to minimize...


## Part 3: Cross-Encoder Re-Ranker

Implement a rerank function using cross-encoder/ms-marco-MiniLM-L-6-v2 to re-score candidate documents.

In [9]:
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Cross-Encoder loaded")

def rerank(query: str, candidates: list[str], top_k: int = 3) -> list[dict]:
    pairs = [[query, doc] for doc in candidates]
    ce_scores = cross_encoder.predict(pairs)
    scored_docs = [
        {"text": doc, "ce_score": float(score), "rank": rank + 1}
        for rank, (doc, score) in enumerate(sorted(zip(candidates, ce_scores),
                                                     key=lambda x: x[1], reverse=True))
    ]
    return scored_docs[:top_k]


test_candidates = [
    "Transformers use self-attention mechanisms to process sequences in parallel.",
    "The BM25 algorithm ranks documents using TF-IDF with term frequency saturation.",
    "Neural networks learn by adjusting weights through backpropagation.",
]

test_rerank = rerank("How do transformers work?", test_candidates, top_k=2)
print(f"\nTest Re-ranking for: 'How do transformers work?'")
print("-" * 100)
for r in test_rerank:
    print(f"  [#{r['rank']}] CE_score={r['ce_score']:.4f}")
    print(f"    → {r['text'][:90]}...")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Cross-Encoder loaded

Test Re-ranking for: 'How do transformers work?'
----------------------------------------------------------------------------------------------------
  [#1] CE_score=6.2494
    → Transformers use self-attention mechanisms to process sequences in parallel....
  [#2] CE_score=-10.9481
    → Neural networks learn by adjusting weights through backpropagation....


## Part 4: Query Expansion with HyDE

Implement HyDE (Hypothetical Document Embedding) to expand user queries.

**How it works:**
1. User submits a short, vague query
2. LLM generates a hypothetical detailed answer
3. Use the hypothetical doc as the retrieval query
4. Result: Better retrieval through enriched query semantics

In [10]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0)

hyde_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a technical AI tutor. Generate a single factual paragraph (3-5 sentences) that would "
     "directly answer the following question. Write it as if it were an excerpt from a textbook. "
     "Use precise technical terminology."),
    ("human", "{query}")
])

hyde_chain = hyde_prompt | llm | StrOutputParser()

def expand_query_hyde(query: str) -> str:
    return hyde_chain.invoke({"query": query})


test_query = "What is gradient descent?"
print(f"Original Query: '{test_query}'")
print("\nGenerating hypothetical document...")

expanded = expand_query_hyde(test_query)
print(f"\nExpanded Query (HyDE output):\n{expanded}")

Original Query: 'What is gradient descent?'

Generating hypothetical document...

Expanded Query (HyDE output):
Gradient descent is an iterative first-order optimization algorithm widely employed to minimize a differentiable cost function. It operates by iteratively adjusting a model's parameters in the direction opposite to the gradient of the cost function with respect to those parameters. At each step, the algorithm calculates the gradient, which indicates the direction of the steepest ascent, and then updates the parameters by subtracting a fraction of this gradient, scaled by a learning rate, to move towards the function's local or global minimum. This process continues until convergence, typically when the change in the cost function or parameters falls below a predefined threshold, making it fundamental for training various machine learning models.


## Part 5: End-to-End Advanced RAG Pipeline

Wire everything together: Query Expansion → Hybrid Retrieval → Re-Ranking → LLM Generation

In [11]:
generation_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a knowledgeable AI assistant specializing in AI/ML topics. "
     "Answer the user's question using ONLY the provided context documents. "
     "If the answer is not available in the context, say: 'I don't have enough information in the provided documents to answer this.' "
     "Be concise, accurate, and cite which document(s) you used if relevant.\n\n"
     "Context:\n{context}"),
    ("human", "{question}")
])

def advanced_rag(user_query: str) -> dict:
    print(f"\n{'='*100}")
    print(f"Advanced RAG Pipeline")
    print(f"{'='*100}")

    print(f"\n[1] QUERY EXPANSION (HyDE)")
    print(f"    Original: '{user_query}'")

    expanded_query = expand_query_hyde(user_query)
    print(f"    Expanded: '{expanded_query[:150]}...")

    print(f"\n[2] HYBRID RETRIEVAL (BM25 + SBERT + RRF)")
    retrieved_results = hybrid_retriever.retrieve(expanded_query, top_k=5)
    retrieved_docs = [r["text"] for r in retrieved_results]

    print(f"    Retrieved {len(retrieved_docs)} candidates:")
    for i, r in enumerate(retrieved_results, 1):
        print(f"      #{i} [doc_{r['doc_id']}] RRF={r['rrf_score']:.6f} | BM25_rank={r['bm25_rank']}, SBERT_rank={r['sbert_rank']}")
        print(f"         → {r['text'][:85]}...")

    print(f"\n[3] CROSS-ENCODER RE-RANKING")
    reranked_results = rerank(user_query, retrieved_docs, top_k=3)
    reranked_docs = [r["text"] for r in reranked_results]

    print(f"    Re-ranked to top-3:")
    for r in reranked_results:
        print(f"      #{r['rank']}] CE_score={r['ce_score']:.4f}")
        print(f"         → {r['text'][:85]}...")

    print(f"\n[4] GENERATION")
    context = "\n\n".join(f"[Document {i+1}]\n{doc}" for i, doc in enumerate(reranked_docs))

    generation_chain = generation_prompt | llm | StrOutputParser()
    final_answer = generation_chain.invoke({"context": context, "question": user_query})

    print(f"    Final Answer:\n")
    print(f"    {final_answer}")

    return {
        "user_query": user_query,
        "expanded_query": expanded_query,
        "retrieved_docs": retrieved_docs,
        "reranked_docs": reranked_docs,
        "final_answer": final_answer
    }


result = advanced_rag("How do transformers encode meaning?")


Advanced RAG Pipeline

[1] QUERY EXPANSION (HyDE)
    Original: 'How do transformers encode meaning?'
    Expanded: 'Transformers encode meaning primarily through their self-attention mechanism, which allows each token in an input sequence to weigh the relevance of e...

[2] HYBRID RETRIEVAL (BM25 + SBERT + RRF)
    Retrieved 5 candidates:
      #1 [doc_0] RRF=0.032266 | BM25_rank=3, SBERT_rank=1
         → Transformers use self-attention mechanisms to process sequences in parallel, enabling...
      #2 [doc_1] RRF=0.032266 | BM25_rank=1, SBERT_rank=3
         → The multi-head attention mechanism in transformers allows the model to attend to diff...
      #3 [doc_2] RRF=0.031514 | BM25_rank=5, SBERT_rank=2
         → Positional encoding in transformer architectures encodes the order of tokens since se...
      #4 [doc_8] RRF=0.031281 | BM25_rank=2, SBERT_rank=6
         → Neural networks learn by adjusting weights through backpropagation, which computes gr...
      #5 [doc_6] RRF=0.03

## Part 6: Naïve RAG Baseline for Comparison

Implement naïve RAG (dense-only retrieval, no expansion, no re-ranking) for comparison.

In [12]:
def naive_rag(user_query: str) -> dict:
    print(f"\n{'='*100}")
    print(f"Naïve RAG Baseline")
    print(f"{'='*100}")
    print(f"\nQuery: '{user_query}'")

    print(f"\n[1] DENSE-ONLY RETRIEVAL (SBERT)")
    query_vec = hybrid_retriever.sbert.encode([user_query], convert_to_numpy=True)[0]
    query_vec = query_vec / np.linalg.norm(query_vec)
    scores = hybrid_retriever.doc_vecs @ query_vec
    top_indices = np.argsort(scores)[::-1][:3]

    print(f"    Top 3 results:")
    for rank, idx in enumerate(top_indices, 1):
        print(f"      #{rank} [doc_{idx}] score={scores[idx]:.4f}")
        print(f"         → {corpus[idx][:85]}...")

    top_doc = corpus[top_indices[0]]

    print(f"\n[2] GENERATION from top-1 document")
    context = f"[Document 1]\n{top_doc}"

    generation_chain = generation_prompt | llm | StrOutputParser()
    final_answer = generation_chain.invoke({"context": context, "question": user_query})

    print(f"    Final Answer:\n")
    print(f"    {final_answer}")

    return {
        "user_query": user_query,
        "top_doc": top_doc,
        "final_answer": final_answer
    }


naive_result = naive_rag("How do transformers encode meaning?")


Naïve RAG Baseline

Query: 'How do transformers encode meaning?'

[1] DENSE-ONLY RETRIEVAL (SBERT)
    Top 3 results:
      #1 [doc_2] score=0.5907
         → Positional encoding in transformer architectures encodes the order of tokens since se...
      #2 [doc_1] score=0.5406
         → The multi-head attention mechanism in transformers allows the model to attend to diff...
      #3 [doc_6] score=0.4742
         → BERT is a bidirectional transformer encoder trained using masked language modeling wh...

[2] GENERATION from top-1 document
    Final Answer:

    I don't have enough information in the provided documents to answer this.


## Part 7: Comparison Experiment

Run 3 test queries through both pipelines and fill in the comparison table.

In [13]:
import pandas as pd

test_queries = [
    "How do transformers encode meaning?",
    "Optimization techniques for training neural networks",
    "What is BM25 and why is it important?"
]

print("Running comparison experiment...")
print("This will run each query through both Naïve and Advanced RAG pipelines.\n")

comparison_data = []

for i, query in enumerate(test_queries, 1):
    print(f"\n{'='*100}")
    print(f"QUERY {i}: '{query}'")
    print(f"{'='*100}")

    print(f"\n--- NAÏVE RAG ---")
    naive_result = naive_rag(query)
    naive_top_doc = naive_result["top_doc"]

    print(f"\n--- ADVANCED RAG ---")
    advanced_result = advanced_rag(query)
    advanced_top_doc = advanced_result["reranked_docs"][0] if advanced_result["reranked_docs"] else "N/A"

    different = naive_top_doc != advanced_top_doc

    comparison_data.append({
        "Query": query,
        "Naïve RAG Top Doc": naive_top_doc[:90] + "..." if len(naive_top_doc) > 90 else naive_top_doc,
        "Advanced RAG Top Doc": advanced_top_doc[:90] + "..." if len(advanced_top_doc) > 90 else advanced_top_doc,
        "Are they different?": "Yes" if different else "No"
    })

comparison_df = pd.DataFrame(comparison_data)

print(f"\n\n{'='*100}")
print(f"COMPARISON TABLE")
print(f"{'='*100}\n")

print(comparison_df.to_string(index=False))

print(f"\n\n{'='*100}")
print(f"MARKDOWN TABLE FORMAT")
print(f"{'='*100}\n")

print("|Query|Naïve RAG Top Doc|Advanced RAG Top Doc|Are they different?|")
print("|---|---|---|---|")
for _, row in comparison_df.iterrows():
    print(f"|{row['Query']}|{row['Naïve RAG Top Doc']}|{row['Advanced RAG Top Doc']}|{row['Are they different?']}|")

Running comparison experiment...
This will run each query through both Naïve and Advanced RAG pipelines.


QUERY 1: 'How do transformers encode meaning?'

--- NAÏVE RAG ---

Naïve RAG Baseline

Query: 'How do transformers encode meaning?'

[1] DENSE-ONLY RETRIEVAL (SBERT)
    Top 3 results:
      #1 [doc_2] score=0.5907
         → Positional encoding in transformer architectures encodes the order of tokens since se...
      #2 [doc_1] score=0.5406
         → The multi-head attention mechanism in transformers allows the model to attend to diff...
      #3 [doc_6] score=0.4742
         → BERT is a bidirectional transformer encoder trained using masked language modeling wh...

[2] GENERATION from top-1 document
    Final Answer:

    I don't have enough information in the provided documents to answer this.

--- ADVANCED RAG ---

Advanced RAG Pipeline

[1] QUERY EXPANSION (HyDE)
    Original: 'How do transformers encode meaning?'
    Expanded: 'Transformers encode meaning primarily through

## Key Observations

### What Each Component Contributes:

1. **Hybrid Retrieval (BM25 + SBERT + RRF):**
   - BM25: Excels at keyword matching (e.g., "BM25", "term frequency")
   - SBERT: Excels at semantic similarity (e.g., "transformers" ≈ "attention mechanisms")
   - RRF: Combines both by fusing ranks, preventing either from dominating

2. **Query Expansion (HyDE):**
   - Transforms short, vague queries into rich, detailed hypothetical answers
   - Helps bridge the gap between user terminology and document vocabulary

3. **Cross-Encoder Re-Ranking:**
   - Reads query + document together (unlike SBERT which encodes them separately)
   - Provides a final precision pass: refines top-20 candidates down to top-3

4. **Why Advanced RAG is Better:**
   - Handles vocabulary mismatch better than naïve dense-only retrieval
   - Recovers documents that SBERT might miss but BM25 would find
   - Re-ranking ensures only the most relevant docs reach the LLM

### Findings:
- For keyword-heavy queries: Advanced RAG finds doc_8 (BM25), Naïve RAG might miss it
- For semantic queries: Advanced RAG uses query expansion to better understand intent
- For mixed queries: Advanced RAG's fusion strategy wins